<a href="https://colab.research.google.com/github/Asonvar/MediGemmaProject/blob/main/MediGemmaProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch accelerate gradio bitsandbytes

In [4]:
import torch
import gradio as gr
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from google.colab import userdata
import traceback

# 1. Fetch your securely stored Hugging Face token
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    raise ValueError("You did not save the HF_TOKEN in the Colab Secrets manager correctly.")

# 2. Configure 4-bit Quantization to fit in the T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 3. Load the Model and Processor
model_id = "google/medgemma-1.5-4b-it"

print("Downloading MedGemma 1.5 (4B) weights... This will take several minutes.")
processor = AutoProcessor.from_pretrained(model_id, token=hf_token)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)
print("Model loaded successfully.")

import traceback

# 4. Define the Inference Logic (Gemma 3 Architecture Format)
def analyze_medical_scan(image, text_prompt):
    try:
        if image is None:
            return "Error: You must upload an image."
        if not text_prompt:
            text_prompt = "Describe the findings in this medical image."

        # Ensure 3-channel RGB for the SigLIP encoder
        image = image.convert("RGB")

        # GEMMA 3 CHAT TEMPLATE FORMAT
        # We must pass the image explicitly inside the content dictionary
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": text_prompt}
                ]
            }
        ]

        # The apply_chat_template function correctly maps the image into the tokenizer's sequence
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        )

        # Move inputs to GPU. The specialized processor .to() method handles mixed dtypes correctly here.
        inputs = inputs.to(model.device, dtype=torch.float16)

        # Generate the response
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256)

        # We must slice the output to remove the prompt tokens, otherwise it repeats the prompt back to us
        input_len = inputs["input_ids"].shape[-1]
        generated_tokens = outputs[0][input_len:]

        response = processor.decode(generated_tokens, skip_special_tokens=True)
        return response

    except Exception as e:
        error_trace = traceback.format_exc()
        return f"BACKEND CRASH DETECTED:\n\n{error_trace}"

# 5. Build the UI
interface = gr.Interface(
    fn=analyze_medical_scan,
    inputs=[
        gr.Image(type="pil", label="Upload Medical Scan"),
        gr.Textbox(lines=2, placeholder="e.g., What pathology is visible?")
    ],
    outputs=gr.Textbox(label="MediGemma-X Output"),
    title="MediGemma-X MVP (Stage 1)",
    description="Basic wrapper testing MedGemma 1.5."
)

interface.launch(share=True)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Model loaded successfully.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2cb570b6d52192db2c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
